In [ ]:
"""
sandbox.ipynb

A sandbox to play around with new analyses.

Author: Stellina X. Ao
Created: 2026-03-05
Last Modified: 2026-03-23
Python Version: 3.11.14
"""

from squiggs.neuron_viewer import NeuronViewer
from squiggs.renderers import FitRenderer
from sg.fitter import LVMFamily
from sg.eval_models import plot_summary

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
"""
TODO:
backburner
- what about adding ReLU to the taskvar model and removing block?
- plot rasters, sanity check the psths
"""

In [ ]:
# check if you can get back m2 z-score psths from dividing tuber
# ask joao about the masking of the problem via zscore

## Fit 

In [ ]:
from sg.data import load_sess

unit_spike_times, trial_data, session_data, regions = load_sess(
    subj_id="MR83", sess_idx=5, mode="new"
)

In [ ]:
family = LVMFamily(
    trial_data=trial_data,
    spike_times=unit_spike_times,
    session_data=session_data,
    regions=regions,
    n_latents_mult=3,
    n_latents_addt=1,
    sanity_check=0,
    task_vars=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
    tpre=0.5,
    tpost=1,
    refit=True,
    max_iter=10,
    norm_activity=True,
)
family.fit_all()
family.eval()

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from sg.data import get_psths_cond, get_choice_ts

renderer = PETHRasterRenderer(
    event_times=get_choice_ts(family.trial_data, mode="response"),
    spike_times=family.spike_times["DMS"],
    peths=get_psths_cond(
        family.psths["DMS"], family.trial_data, family.trial_mask, mode="response"
    ),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
    s=1,
    linewidths=0.5,
)

nv = NeuronViewer(num_units=family.psths["DMS"].shape[0], render_func=renderer)

In [ ]:
renderer = FitRenderer(
    family.mod_affine,
    x=family.test_dl.dataset[:],
    y=family.robs,
    save_subdir=Path("model_fits") / "0312-lm" / "offset",
)

nv = NeuronViewer(num_units=renderer.y.shape[1], render_func=renderer)

In [ ]:
_ = plot_summary(family, model=family.mod_gain, potato=family.strategy, mode="gain")

### Delta R2, R2

In [ ]:
from sg.eval_models import monkey_cv_d_r2, plot_cv_d_r2

scramble_r2s, scramble_r2s_summary = monkey_cv_d_r2(
    trial_data, unit_spike_times, session_data, regions
)
plot_cv_d_r2(scramble_r2s_summary)

### Strategy Decoding from Latents

In [ ]:
from sg.eval_models import latent_decoding, get_scores_mean
from pprint import pprint

scores = latent_decoding(family)
pprint(get_scores_mean(scores))

### cweights

In [ ]:
import matplotlib.colors as colors
import matplotlib.cm as cm

taskvar_str = [
    "resp_L",
    "resp_R",
    "reward_no",
    "reward_yes",
    "block_side_L",
    "block_side_R",
    "p_resp_L",
    "p_resp_no",
    "p_resp_R",
    "p_reward_no",
    "p_reward_yes",
]


def plot_cweights(family, mode, i=0, ax0=0, ax1=1):
    model = family.mod_gain if mode == "gain" else family.mod_offset
    taskvar_weights = family.mod_taskvar.tv.weight.data[:][i]
    norm = colors.Normalize(vmin=taskvar_weights.min(), vmax=taskvar_weights.max())
    cmap = cm.viridis

    mapped_colors = cmap(norm(taskvar_weights))

    coupling = (
        model.readout_gain.weight.data[:].T
        if mode == "gain"
        else mode.readout_offset.weight.data
    )
    fig, ax = plt.subplots(figsize=(3, 3))

    ax.scatter(coupling[:, ax0], coupling[:, ax1], color=mapped_colors, s=0.5)
    fig.suptitle(f"Task Var: {taskvar_str[i]}")


for i in range(len(taskvar_str)):
    plot_cweights(family, mode="gain", ax0=1, ax1=2, i=i)

In [ ]:
from sg.eval_models import plot_cweights_mult

plot_cweights_mult(family)

### res dfs filtering stuff

In [ ]:
from sg.eval_models import get_res

res = get_res(family, mode="taskvar")
radish = np.mean(res, axis=1)
bacon = np.std(res, axis=1)

In [ ]:
# 1. why is res.shape = (332, 150) but dfs.shape = (332, 154)
# -> how is robs filtered?
# 2. is the filtered array (with the offending neurons filtered out) passed to the lvms?
#    -> because it's still learning a major affect present in only 6 neurons

In [ ]:
res.shape

In [ ]:
np.where(np.abs(res[149, :]) > 5)

In [ ]:
plt.figure()
plt.hist(res[149, :])
plt.show()

In [ ]:
family.sample["dfs"][
    :, family.cids
].shape  # = robs.shape, makes sense beceause robs is filtered with cids

In [ ]:
plt.figure()
plt.plot(radish)
plt.fill_between(np.arange(len(radish)), radish - bacon, radish + bacon, alpha=0.5)
plt.show()

In [ ]:
plt.figure()
plt.imshow(family.data_gd.covariates["dfs"])
plt.show()

In [ ]:
_ = plot_summary(family, model=family.mod_offset, potato=bacon, mode="offset")

In [ ]:
plt.figure()
plt.imshow(family.robs.T, aspect="auto")
plt.show()